# Model Optimization

This notebook focuses on improving the baseline models developed in the previous stage.

While the baseline models provided an initial benchmark, further improvements can be achieved through hyperparameter tuning and testing additional algorithms. The goal is to enhance predictive performance while maintaining model generalization.

Optimization experiments in this notebook will include:

* Hyperparameter tuning for the Random Forest classifier
* Evaluation using time-aware validation strategies
* Comparison with other ensemble models where appropriate

The results will be compared against the baseline models established earlier.


## Import Required Libraries

The necessary libraries for model optimization are imported. These include tools for data handling, model training, and hyperparameter tuning using cross-validation.


In [36]:
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import accuracy_score, classification_report

## Load Engineered Dataset

The engineered dataset generated during the feature engineering stage is loaded. This dataset contains the cleaned and transformed features used for training the machine learning models.


In [37]:
df = pd.read_csv("../data/processed_data/fashion_trend_engineered.csv")

print(df.shape)

df.head()

(1416, 37)


,week,year,month,search_index,search_growth,momentum_4w,keyword_frequency,brand_popularity,avg_price,avg_rating,...,season_Monsoon,season_Summer,season_Winter,category_Jeans,category_Kurti,category_Saree,category_Streetwear,category_TShirt,trend_label,next_week_search
0,4,2021,1,55.04,0.125793,52.5100,153,0.54,1970.19,3.29,...,0,0,1,0,0,0,0,0,2,56.72
1,5,2021,2,56.72,0.030523,53.8450,150,0.46,1758.52,4.40,...,0,0,1,0,0,0,0,0,1,62.65
2,6,2021,2,62.65,0.104549,55.8250,189,0.89,2190.03,4.72,...,0,0,1,0,0,0,0,0,2,58.93
3,7,2021,2,58.93,-0.059377,58.3350,186,0.46,1803.50,4.03,...,0,0,1,0,0,0,0,0,0,64.61
4,8,2021,2,64.61,0.096386,60.7275,201,0.84,1884.70,3.32,...,0,0,1,0,0,0,0,0,2,63.18


## Prepare Features and Target Variables

For the classification task, the target variable is `trend_label`.
All remaining variables are used as input features.

However, features that previously caused **target leakage** (`search_growth`, `search_index`, `momentum_4w`) are excluded to ensure the model learns meaningful predictive patterns.


In [38]:
X = df.drop(columns=[
    "trend_label",
    "next_week_search",
    "search_growth",
    "search_index",
    "momentum_4w"
])

y = df["trend_label"]

print("Feature shape:", X.shape)

Feature shape: (1416, 32)


## Time-Based Train-Test Split

Because the dataset represents weekly time-series observations, a chronological split is used. Earlier observations are used for training and later observations are used for testing to avoid information leakage from future data.


In [39]:
split_index = int(len(X) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (1132, 32)
Test size: (284, 32)


## Time-Series Cross Validation

Instead of random cross-validation, TimeSeriesSplit is used because the dataset is time-dependent. This method ensures that training data always comes from earlier time periods than validation data.


In [40]:
tscv = TimeSeriesSplit(n_splits=5)

tscv

TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None)

## Hyperparameter Tuning with GridSearchCV

To improve the performance of the Random Forest classifier, hyperparameter tuning is performed using GridSearchCV. The search explores multiple combinations of parameters such as the number of trees, tree depth, and minimum samples required for splits. TimeSeriesSplit is used to maintain temporal order during validation.


In [41]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

rf = RandomForestClassifier(random_state=42)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=tscv,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)

Best Parameters: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Best CV Score: 0.6297872340425532


## Evaluate Tuned Model on Test Data

After identifying the best hyperparameters using GridSearchCV, the optimized Random Forest model is evaluated on the held-out test dataset. This provides an unbiased estimate of the model’s predictive performance on unseen data.


In [42]:
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Test Accuracy: 0.6514084507042254

Classification Report:

              precision    recall  f1-score   support

           0       0.69      0.52      0.59        63
           1       0.61      0.86      0.72       142
           2       0.81      0.38      0.52        79

    accuracy                           0.65       284
   macro avg       0.70      0.59      0.61       284
weighted avg       0.68      0.65      0.63       284



tuning did not improve accuracy, which means your baseline RandomForest was already near optimal.

## Train Gradient Boosting Classifier

Gradient Boosting is another ensemble learning method that builds trees sequentially, where each new tree corrects the errors made by previous trees. It often performs well on structured/tabular datasets and can sometimes outperform Random Forest models.


In [43]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(random_state=42)

gb_model.fit(X_train, y_train)

y_pred_gb = gb_model.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred_gb))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_gb))

Test Accuracy: 0.7288732394366197

Classification Report:

              precision    recall  f1-score   support

           0       0.69      0.70      0.69        63
           1       0.72      0.80      0.76       142
           2       0.79      0.62      0.70        79

    accuracy                           0.73       284
   macro avg       0.73      0.71      0.72       284
weighted avg       0.73      0.73      0.73       284



## Save Best Performing Model

After comparing multiple models, the Gradient Boosting classifier achieved the best predictive performance. The trained model is saved so it can be reused later for deployment or inference without retraining.


In [44]:
import joblib

joblib.dump(gb_model, "../models/trend_classifier.pkl")

print("Model saved successfully.")

Model saved successfully.
